In [0]:
## Unity Catalog context (Azure)
spark.sql('USE CATALOG projeto_data_engineering')
spark.sql('USE SCHEMA kabum_project')

# Storage account name (adjust if yours is different)
STORAGE_ACCOUNT = 'storagekabumdata'
BRONZE_ABFSS = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
SILVER_ABFSS = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_ABFSS   = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/"


In [0]:
%sql
-- GOLD_FEATURES_V2_SCORED_PATH
-- /Volumes/workspace/kabum_project/lake/gold/kabum/notebooks_features_enriched_scored_delta

-- GOLD_QUALITY_DAILY_PATH
-- /Volumes/workspace/kabum_project/lake/gold/kabum/catalog_quality_daily_delta

Views para o dashboard

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_kabum_features_scored AS
SELECT *
FROM projeto_data_engineering.kabum_project.notebooks_features_scored;

CREATE OR REPLACE TEMP VIEW v_kabum_quality_daily AS
SELECT *
FROM projeto_data_engineering.kabum_project.catalog_quality_daily;

KPIs do último dia disponível

In [0]:
%sql
WITH last_day AS (
  SELECT max(ingestion_date) AS ingestion_date
  FROM v_kabum_quality_daily
)
SELECT
  q.ingestion_date,
  q.search_term,
  q.marketplace,
  q.products,
  q.avg_completeness_pct,
  q.median_completeness_pct,
  q.high_pct,
  q.medium_pct,
  q.low_pct
FROM v_kabum_quality_daily q
JOIN last_day d
  ON q.ingestion_date = d.ingestion_date
ORDER BY q.search_term, q.marketplace;

ingestion_date,search_term,marketplace,products,avg_completeness_pct,median_completeness_pct,high_pct,medium_pct,low_pct
2026-03-05,notebook,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook gamer,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook i5,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook ryzen,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,ultrabook,kabum,40,100.0,100.0,100.0,0.0,0.0


KPIs filtráveis por período (para “trend cards”)

In [0]:
%sql
SELECT
  ingestion_date,
  search_term,
  marketplace,
  products,
  avg_completeness_pct,
  median_completeness_pct,
  high_pct,
  medium_pct,
  low_pct
FROM v_kabum_quality_daily
ORDER BY ingestion_date DESC, search_term, marketplace;

ingestion_date,search_term,marketplace,products,avg_completeness_pct,median_completeness_pct,high_pct,medium_pct,low_pct
2026-03-05,notebook,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook gamer,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook i5,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook ryzen,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,ultrabook,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-04,notebook,kabum,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,notebook gamer,kabum,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,notebook i5,kabum,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,notebook ryzen,kabum,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,ultrabook,kabum,20,100.0,100.0,100.0,0.0,0.0


Avg completeness ao longo do tempo

In [0]:
%sql
SELECT
  ingestion_date,
  search_term,
  marketplace,
  avg_completeness_pct
FROM v_kabum_quality_daily
ORDER BY ingestion_date, search_term, marketplace;

ingestion_date,search_term,marketplace,avg_completeness_pct
2026-03-04,notebook,kabum,100.0
2026-03-04,notebook gamer,kabum,100.0
2026-03-04,notebook i5,kabum,100.0
2026-03-04,notebook ryzen,kabum,100.0
2026-03-04,ultrabook,kabum,100.0
2026-03-05,notebook,kabum,100.0
2026-03-05,notebook gamer,kabum,100.0
2026-03-05,notebook i5,kabum,100.0
2026-03-05,notebook ryzen,kabum,100.0
2026-03-05,ultrabook,kabum,100.0


Stacked % tiers ao longo do tempo

In [0]:
%sql
SELECT
  ingestion_date,
  search_term,
  marketplace,
  high_pct,
  medium_pct,
  low_pct
FROM v_kabum_quality_daily
ORDER BY ingestion_date, search_term, marketplace;

ingestion_date,search_term,marketplace,high_pct,medium_pct,low_pct
2026-03-04,notebook,kabum,100.0,0.0,0.0
2026-03-04,notebook gamer,kabum,100.0,0.0,0.0
2026-03-04,notebook i5,kabum,100.0,0.0,0.0
2026-03-04,notebook ryzen,kabum,100.0,0.0,0.0
2026-03-04,ultrabook,kabum,100.0,0.0,0.0
2026-03-05,notebook,kabum,100.0,0.0,0.0
2026-03-05,notebook gamer,kabum,100.0,0.0,0.0
2026-03-05,notebook i5,kabum,100.0,0.0,0.0
2026-03-05,notebook ryzen,kabum,100.0,0.0,0.0
2026-03-05,ultrabook,kabum,100.0,0.0,0.0


Distribuição de score (histograma)

In [0]:
%sql
SELECT
  catalog_completeness_score,
  count(*) AS products
FROM v_kabum_features_scored
GROUP BY catalog_completeness_score
ORDER BY catalog_completeness_score;

catalog_completeness_score,products
3,200


Fill-rate por coluna (qualidade por spec)

In [0]:
%sql
WITH base AS (
  SELECT
    price,
    old_price,
    discount_pct,
    rating,
    reviews_count,
    refresh_rate_hz_scraped,
    screen_resolution_scraped,
    panel_type_scraped,
    panel_finish_scraped,
    brightness_nits_scraped,
    product_url_scraped
  FROM v_kabum_features_scored
)
SELECT 'price' AS field,
       AVG(CASE WHEN price IS NOT NULL THEN 1.0 ELSE 0.0 END) AS fill_rate
FROM base
UNION ALL
SELECT 'old_price',
       AVG(CASE WHEN old_price IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'discount_pct',
       AVG(CASE WHEN discount_pct IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'rating',
       AVG(CASE WHEN rating IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'reviews_count',
       AVG(CASE WHEN reviews_count IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'refresh_rate_hz_scraped',
       AVG(CASE WHEN refresh_rate_hz_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'screen_resolution_scraped',
       AVG(CASE WHEN screen_resolution_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'panel_type_scraped',
       AVG(CASE WHEN panel_type_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'panel_finish_scraped',
       AVG(CASE WHEN panel_finish_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'brightness_nits_scraped',
       AVG(CASE WHEN brightness_nits_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'product_url_scraped',
       AVG(CASE WHEN product_url_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base;

field,fill_rate
price,1.00000
old_price,0.51000
discount_pct,1.00000
rating,1.00000
reviews_count,1.00000
refresh_rate_hz_scraped,0.79000
screen_resolution_scraped,0.97000
panel_type_scraped,0.72000
panel_finish_scraped,0.78000
brightness_nits_scraped,0.60000


Produtos com LOW completeness (top exemplos)

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  catalog_completeness_score,
  catalog_completeness_pct,
  catalog_completeness_tier,
  refresh_rate_hz_scraped,
  screen_resolution_scraped,
  panel_type_scraped,
  panel_finish_scraped,
  brightness_nits_scraped,
  product_url
FROM v_kabum_features_scored
WHERE catalog_completeness_tier = 'LOW'
ORDER BY catalog_completeness_pct ASC, product_name
LIMIT 50;


product_name,brand,price,rating,reviews_count,catalog_completeness_score,catalog_completeness_pct,catalog_completeness_tier,refresh_rate_hz_scraped,screen_resolution_scraped,panel_type_scraped,panel_finish_scraped,brightness_nits_scraped,product_url


Gamer sem refresh_rate (flag)

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  search_term,
  refresh_rate_hz_scraped,
  screen_resolution_scraped,
  panel_type_scraped,
  product_url
FROM v_kabum_features_scored
WHERE lower(search_term) LIKE '%gamer%'
  AND refresh_rate_hz_scraped IS NULL
ORDER BY product_name
LIMIT 50;


product_name,brand,price,search_term,refresh_rate_hz_scraped,screen_resolution_scraped,panel_type_scraped,product_url
"Notebook Gamer Gigabyte AORUS 5 Intel Core i7-12700H, 16GB RAM, RTX 3060, SSD 1TB, 15.6"" FHD, Windows 11 Home, Preto - KE4-72BR314SH",Gigabyte,9999.0,notebook gamer,null,3840x2160,OLED,https://www.kabum.com.br/produto/420222/notebook-gamer-gigabyte-aorus-5-intel-core-i7-12700h-16gb-ram-rtx-3060-ssd-1tb-15-6-fhd-windows-11-home-preto-ke4-72br314sh
"Notebook Gamer Gigabyte AORUS 5 Intel Core i7-12700H, 16GB RAM, RTX 3060, SSD 1TB, 15.6"" FHD, Windows 11 Home, Preto - KE4-72BR314SH",Gigabyte,9999.0,notebook gamer,null,3840x2160,OLED,https://www.kabum.com.br/produto/420222/notebook-gamer-gigabyte-aorus-5-intel-core-i7-12700h-16gb-ram-rtx-3060-ssd-1tb-15-6-fhd-windows-11-home-preto-ke4-72br314sh


Fonte do painel (TITLE/RAW/MISSING)

In [0]:
%sql
SELECT
  panel_type_scraped AS panel_type,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM v_kabum_features_scored
GROUP BY panel_type_scraped
ORDER BY products DESC;


panel_type,products,pct
IPS,104,52.00
null,56,28.00
TN,24,12.00
OLED,14,7.00
VA,2,1.00


Distribuição por Tipo de Painel (Market Mix)

In [0]:
%sql
SELECT
  panel_type_scraped AS panel_type,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM v_kabum_features_scored
GROUP BY panel_type_scraped
ORDER BY products DESC;


panel_type,products,pct
IPS,104,52.00
null,56,28.00
TN,24,12.00
OLED,14,7.00
VA,2,1.00


Distribuição de Refresh Rate

In [0]:
%sql
SELECT
  refresh_rate_hz_scraped AS refresh_rate_hz,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY refresh_rate_hz_scraped
ORDER BY refresh_rate_hz_scraped;


refresh_rate_hz,products
null,42
60.0,54
120.0,16
144.0,44
165.0,24
240.0,20


Gamer vs Não Gamer

In [0]:
%sql
SELECT
  CASE 
    WHEN lower(product_name) RLIKE '(gamer|aorus|nitro|legion|predator|tuf|rog|strix|omen|alienware)'
    THEN 'GAMER'
    ELSE 'NON_GAMER'
  END AS segment,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY segment;

segment,products
GAMER,70
NON_GAMER,130


Distribuição de Tamanho de Tela

In [0]:
%sql
SELECT
  category,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY category
ORDER BY products DESC
LIMIT 20;


category,products
Computadores/Notebooks/Notebook Lenovo,44
Computadores/Notebooks/Ultrabook,40
Computadores/Notebooks/Notebook Gamer/Notebook Gamer Acer,28
Computadores/Notebooks/Notebook Asus,24
Computadores/Notebooks/Notebook Acer,22
Computadores/Notebooks/Notebook Gamer/Notebook Gamer Asus,18
Computadores/Notebooks/Notebook Gamer,8
Computadores/Notebooks/Notebook Vaio,6
Computadores/Notebooks/Notebook Samsung,2
Computadores/Notebooks/Notebook Dell,2


Top CPUs (Market Intelligence)

In [0]:
%sql
SELECT
  search_term,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY search_term
ORDER BY products DESC;


search_term,products
ultrabook,40
notebook i5,40
notebook,40
notebook gamer,40
notebook ryzen,40


Completeness vs Categoria Gamer

In [0]:
%sql
SELECT
  CASE 
    WHEN lower(product_name) RLIKE '(gamer|aorus|nitro|legion|predator|tuf|rog|strix|omen|alienware)'
    THEN 'GAMER'
    ELSE 'NON_GAMER'
  END AS segment,
  AVG(catalog_completeness_pct) AS avg_completeness
FROM v_kabum_features_scored
GROUP BY segment;

segment,avg_completeness
GAMER,100.0
NON_GAMER,100.0


Distribuição de Storage

In [0]:
%sql
SELECT
  CAST(brightness_nits_scraped AS DOUBLE) AS brightness_nits,
  COUNT(*) AS products
FROM v_kabum_features_scored
WHERE brightness_nits_scraped IS NOT NULL
GROUP BY CAST(brightness_nits_scraped AS DOUBLE)
ORDER BY brightness_nits;


brightness_nits,products
220.0,20
250.0,26
300.0,48
370.0,2
400.0,6
500.0,18


KPI executivo do “último dia”

In [0]:
%sql
WITH last_day AS (
  SELECT max(ingestion_date) AS ingestion_date
  FROM v_kabum_quality_daily
)
SELECT
  q.ingestion_date,
  q.search_term,
  q.marketplace,
  q.products,
  q.avg_completeness_pct,
  q.median_completeness_pct,
  q.high_pct,
  q.medium_pct,
  q.low_pct
FROM v_kabum_quality_daily q
JOIN last_day d
  ON q.ingestion_date = d.ingestion_date
ORDER BY q.search_term, q.marketplace;

ingestion_date,search_term,marketplace,products,avg_completeness_pct,median_completeness_pct,high_pct,medium_pct,low_pct
2026-03-05,notebook,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook gamer,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook i5,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,notebook ryzen,kabum,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,ultrabook,kabum,40,100.0,100.0,100.0,0.0,0.0


Marca líder do catálogo (presença)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_catalog
FROM v_kabum_features_scored
GROUP BY brand
ORDER BY products DESC
LIMIT 20;

brand,products,pct_of_catalog
ASUS,62,31.00
Acer,56,28.00
Lenovo,44,22.00
Samsung,8,4.00
MSI,8,4.00
VAIO,6,3.00
Microsoft,4,2.00
Hp,4,2.00
Gigabyte,4,2.00
Dell,2,1.00


Ticket médio por marca (preço atual)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(AVG(price), 2) AS avg_price,
  ROUND(percentile_approx(price, 0.5), 2) AS median_price
FROM v_kabum_features_scored
WHERE price IS NOT NULL
GROUP BY brand
HAVING COUNT(*) >= 3
ORDER BY avg_price DESC;

brand,products,avg_price,median_price
MSI,8,30950.0,30900.0
Microsoft,4,21550.0,18700.0
Samsung,8,20523.51,17000.0
Gigabyte,4,13149.5,9999.0
ASUS,62,12926.55,6447.0
Acer,56,6496.51,5760.0
Lenovo,44,4280.74,3999.0
VAIO,6,3877.4,3284.1
Hp,4,2806.25,2199.0


“Valor de portfólio” por marca (proxy de potencial de receita)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(SUM(price), 2) AS total_catalog_value
FROM v_kabum_features_scored
WHERE price IS NOT NULL
GROUP BY brand
ORDER BY total_catalog_value DESC;

brand,products,total_catalog_value
ASUS,62,801446.08
Acer,56,363804.66
MSI,8,247600.0
Lenovo,44,188352.64
Samsung,8,164188.1
Microsoft,4,86200.0
Gigabyte,4,52598.0
Apple,2,23998.0
VAIO,6,23264.4
Hp,4,11224.98


KPI de desconto (média e mediana por marca)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct,
  ROUND(percentile_approx(discount_pct, 0.5), 2) AS median_discount_pct
FROM v_kabum_features_scored
WHERE discount_pct IS NOT NULL
GROUP BY brand
HAVING COUNT(*) >= 3
ORDER BY avg_discount_pct DESC;

brand,products,avg_discount_pct,median_discount_pct
Acer,56,6.43,10
ASUS,62,6.16,10
Lenovo,44,6.05,10
Gigabyte,4,5.0,0
VAIO,6,3.33,0
Microsoft,4,0.0,0
Samsung,8,0.0,0
MSI,8,0.0,0
Hp,4,0.0,0


Desconto real em R$ (melhores “deals”)

In [0]:
%sql
SELECT
  product_name,
  brand,
  ROUND(old_price, 2) AS old_price,
  ROUND(price, 2) AS price,
  ROUND(old_price - price, 2) AS discount_brl,
  ROUND(discount_pct, 2) AS discount_pct
FROM v_kabum_features_scored
WHERE price IS NOT NULL
  AND old_price IS NOT NULL
  AND old_price >= price
ORDER BY discount_brl DESC
LIMIT 50;


product_name,brand,old_price,price,discount_brl,discount_pct
"Notebook Gamer Rog Strix G16 G615jpr Nvidia RTX 5070 Intel Core i9 14900hx 32gb Ram 1TB SSD WINDOWS 11 Home Tela 16"" 240hz LED Cinza - S5001w",ASUS,19598.91,17443.03,2155.88,11
"Notebook Gamer Rog Strix G16 G615jpr Nvidia RTX 5070 Intel Core i9 14900hx 32gb Ram 1TB SSD WINDOWS 11 Home Tela 16"" 240hz LED Cinza - S5001w",ASUS,19598.91,17443.03,2155.88,11
"Notebook Gamer ASUS Rog Strix G16 Intel Core I7, 8gb Ram, GeForce Rtx 4050, Ssd 512gb, 16"" Fhd, 165hz, Keepos, Eclipse Gray - G614ju-N3367",ASUS,12940.0,10999.0,1941.0,15
"Notebook Gamer ASUS Rog Strix G16 Intel Core I7, 8gb Ram, GeForce Rtx 4050, Ssd 512gb, 16"" Fhd, 165hz, Keepos, Eclipse Gray - G614ju-N3367",ASUS,12940.0,10999.0,1941.0,15
"Notebook Gamer Predator Helios Neo 16 AI PHN16-73-76TF, Intel® Core™ Ultra 7 255HX, 16GB RAM, 512GB SSD, RTX 5070, WQXGA 16""",Acer,18999.0,17099.1,1899.9,10
"Notebook Gamer Predator Helios Neo 16 AI PHN16-73-76TF, Intel® Core™ Ultra 7 255HX, 16GB RAM, 512GB SSD, RTX 5070, WQXGA 16""",Acer,18999.0,17099.1,1899.9,10
Notebook Acer Predator Phn16-72-99my Intel Ci9 14900hx 14ª Gen 32gb 1TB SSD RTX 4070 W11 Home - NH.QTVAL.002,Acer,14999.0,13499.1,1499.9,10
Notebook Acer Predator Phn16-72-99my Intel Ci9 14900hx 14ª Gen 32gb 1TB SSD RTX 4070 W11 Home - NH.QTVAL.002,Acer,14999.0,13499.1,1499.9,10
"Notebook Asus Tuf Gaming F16 Fx608jhr Nvidia RTX 5050 Intel Core i7 14650hx 16gb Ram 512gb SSD WINDOWS 11 Home Tela 16"" 165hz Nível Ips Cinza - Rv016w",ASUS,11671.85,10387.95,1283.9,11
"Notebook Asus Tuf Gaming F16 Fx608jhr Nvidia RTX 5050 Intel Core i7 14650hx 16gb Ram 512gb SSD WINDOWS 11 Home Tela 16"" 165hz Nível Ips Cinza - Rv016w",ASUS,11671.85,10387.95,1283.9,11


Segmento Gamer vs Não-Gamer (mix + ticket + desconto)

In [0]:
%sql
WITH base AS (
  SELECT
    CASE 
      WHEN lower(product_name) RLIKE '(gamer|aorus|nitro|legion|predator|tuf|rog|strix|omen|katana|victus|loq|helios|zephyrus|alienware|ideapad gaming)'
      THEN 'GAMER'
      ELSE 'NON_GAMER'
    END AS segment,
    price,
    discount_pct
  FROM v_kabum_features_scored
)
SELECT
  segment,
  COUNT(*) AS products,
  ROUND(AVG(price), 2) AS avg_price,
  ROUND(percentile_approx(price, 0.5), 2) AS median_price,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct
FROM base
GROUP BY segment
ORDER BY products DESC;

segment,products,avg_price,median_price,avg_discount_pct
NON_GAMER,122,8889.21,3960.0,4.49
GAMER,78,11353.48,6908.01,6.41


Distribuição de preço (faixas)

In [0]:
%sql
SELECT
  CASE
    WHEN price < 2000 THEN '< 2k'
    WHEN price < 3000 THEN '2k-3k'
    WHEN price < 4000 THEN '3k-4k'
    WHEN price < 5000 THEN '4k-5k'
    WHEN price < 7000 THEN '5k-7k'
    WHEN price < 10000 THEN '7k-10k'
    ELSE '>= 10k'
  END AS price_bucket,
  COUNT(*) AS products
FROM v_kabum_features_scored
WHERE price IS NOT NULL
GROUP BY price_bucket
ORDER BY products DESC;

price_bucket,products
>= 10k,52
3k-4k,50
5k-7k,48
2k-3k,20
7k-10k,16
4k-5k,14


“Premium share” por marca (ex: % acima de 6k)

In [0]:
%sql
WITH b AS (
  SELECT brand, price
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL
)
SELECT
  brand,
  COUNT(*) AS products,
  SUM(CASE WHEN price >= 6000 THEN 1 ELSE 0 END) AS premium_cnt,
  ROUND(SUM(CASE WHEN price >= 6000 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS premium_pct
FROM b
GROUP BY brand
HAVING COUNT(*) >= 3
ORDER BY premium_pct DESC;

brand,products,premium_cnt,premium_pct
MSI,8,8,100.00
Gigabyte,4,4,100.00
Microsoft,4,4,100.00
Samsung,8,6,75.00
ASUS,62,32,51.61
Acer,56,22,39.29
Lenovo,44,6,13.64
VAIO,6,0,0.00
Hp,4,0,0.00


Value For Money Score

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  refresh_rate_hz_scraped,
  brightness_nits_scraped,

  ROUND(
    (
      COALESCE(CAST(rating AS DOUBLE), 0.0) +
      COALESCE(CAST(refresh_rate_hz_scraped AS DOUBLE), 0.0) / 60.0 +
      COALESCE(CAST(brightness_nits_scraped AS DOUBLE), 0.0) / 300.0 +
      COALESCE(CAST(reviews_count AS DOUBLE), 0.0) / 1000.0
    ) / CAST(price AS DOUBLE)
  , 8) AS value_score

FROM v_kabum_features_scored
WHERE price IS NOT NULL
ORDER BY value_score DESC
LIMIT 20;


product_name,brand,price,rating,reviews_count,refresh_rate_hz_scraped,brightness_nits_scraped,value_score
Notebook Lenovo Ideapad 1 R3-7320u 4gb 256gb SSD Linux 15.6´´ 82x5s00600 Cloud Grey,Lenovo,2471.51,5.0,2,60.0,220.0,0.00272519
Notebook Lenovo Ideapad 1 R3-7320u 4gb 256gb SSD Linux 15.6´´ 82x5s00600 Cloud Grey,Lenovo,2471.51,5.0,2,60.0,220.0,0.00272519
"Notebook Lenovo Ideapad 1 15amn7 Amd Ryzen 3 7320u 8GB 256gb SSD WINDOWS 11 15.6"" - 82x5000mbr Cloud Grey",Lenovo,2966.01,5.0,1,60.0,220.0,0.0022705
"Notebook Lenovo Ideapad 1 15amn7 Amd Ryzen 3 7320u 8GB 256gb SSD WINDOWS 11 15.6"" - 82x5000mbr Cloud Grey",Lenovo,2966.01,5.0,1,60.0,220.0,0.0022705
"Notebook Lenovo Ideapad 1 15amn7 Amd Ryzen 3 7320u 8GB 256gb SSD WINDOWS 11 15.6"" - 82x5000mbr Cloud Grey",Lenovo,2966.01,5.0,1,60.0,220.0,0.0022705
"Notebook Lenovo Ideapad 1 15amn7 Amd Ryzen 3 7320u 8GB 256gb SSD WINDOWS 11 15.6"" - 82x5000mbr Cloud Grey",Lenovo,2966.01,5.0,1,60.0,220.0,0.0022705
"Notebook Acer Aspire Go 15, Intel Core i5-13420h De 13ªG, 8GB RAM, SSD 512GB, Windows 11 Home - Ag15-71p-58md",Acer,3419.1,5.0,1,60.0,220.0,0.00196962
"Notebook Acer Aspire Go 15, Intel Core i5-13420h De 13ªG, 8GB RAM, SSD 512GB, Windows 11 Home - Ag15-71p-58md",Acer,3419.1,5.0,1,60.0,220.0,0.00196962
"Notebook Acer Aspire 5 AMD Ryzen7-5700U, 16GB RAM, SSD 512GB, 15.6"" FHD IPS, AMD Radeon, Linux, Prata - A515-45-R74N",Acer,3699.0,5.0,248,60.0,250.0,0.00191439
"Notebook Acer Aspire 5 AMD Ryzen7-5700U, 16GB RAM, SSD 512GB, 15.6"" FHD IPS, AMD Radeon, Linux, Prata - A515-45-R74N",Acer,3699.0,5.0,248,60.0,250.0,0.00191439


Criar o score de performance

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  refresh_rate_hz_scraped,
  brightness_nits_scraped,

  (
    COALESCE(CAST(rating AS DOUBLE), 0.0) +
    COALESCE(CAST(refresh_rate_hz_scraped AS DOUBLE), 0.0) / 60.0 +
    COALESCE(CAST(brightness_nits_scraped AS DOUBLE), 0.0) / 300.0 +
    COALESCE(CAST(reviews_count AS DOUBLE), 0.0) / 1000.0
  ) AS performance_score

FROM v_kabum_features_scored
WHERE price IS NOT NULL;


product_name,brand,price,rating,reviews_count,refresh_rate_hz_scraped,brightness_nits_scraped,performance_score
"Notebook Lenovo Ideapad Slim 3 15irh10 Intel Core i5-13420h 16gb 512gb SSD WINDOWS 11 15.3"" - 83ns0001br Luna Grey",Lenovo,3695.12,5.0,3,60.0,300.0,7.003
Notebook Asus Vivobook 16 X1605va Intel Core I7 1355u 16gb Ram 512gb SSD WINDOWS 11 Home Tela 16” Ips Fhd Silver - Mb763w,ASUS,4304.81,5.0,1,null,null,5.001
"Notebook Lenovo Loq-e 15iax9e Intel Core i7-12650hx 16gb 512gb SSD RTX 4050 Linux 15.6"" - 83mes00200 Luna Grey",Lenovo,7202.6,5.0,2,144.0,300.0,8.402000000000001
"Notebook Lenovo LOQ-E Core i5-12450HX, RTX 3050, 16GB, 512GB SSD, 15.6"" FHD, IPS, 144Hz, Win 11, Luna Cinza- 83ME0007BR",Lenovo,5947.53,5.0,73,144.0,300.0,8.473
"Notebook Acer Aspire Go 15 AG15-71P-5939 Intel core I5 13ª Geração, 8GB RAM, 256GB SSD, FHD TN Windows 11 Home - NX.JF6AL.004",Acer,3960.0,5.0,1,60.0,220.0,6.734333333333334
"Notebook Gamer Lenovo LOQ, i5-12450HX, 16GB DDR5, 1TB SSD, RTX 4050 6GB, 15.6"" FHD, W11, Cinza - 83ME0001BR",Lenovo,6399.0,5.0,4,144.0,300.0,8.404
"Notebook Lenovo Slim, Ryzen 7 7735HS, 8GB DDR5, 512GB SSD, 15.3"" WUXGA, W11, Cinza - 83MM0004BO",Lenovo,3999.0,5.0,31,60.0,300.0,7.031
"Notebook Lenovo IdeaPad Slim 3i Core I5 13420H, 8GB RAM, 512GB Win11 - 83NS0002BR",Lenovo,4207.8,5.0,2,null,250.0,5.835333333333333
"Notebook Acer Aspire 5 A515-45-R478 AMD Ryzen 5- 5500U, 16GB RAM, 512GB SSD, 15,6"""" FHD Led, IPS, 60Hz, Linux Gutta - NX.AYDAL.00R",Acer,4017.6,5.0,10,60.0,250.0,6.843333333333333
"Notebook Asus Vivobook Go 15 E1504fa Amd Ryzen™ 5 7520u 8GB Ram 256gb SSD Linux Keepos 15,60"" Fhd Mixed Black - Nj731",ASUS,2638.09,4.0,23,null,null,4.023


Query final para o gráfico

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  refresh_rate_hz_scraped,
  brightness_nits_scraped,

  (
    COALESCE(CAST(rating AS DOUBLE), 0.0) +
    COALESCE(CAST(refresh_rate_hz_scraped AS DOUBLE), 0.0) / 60.0 +
    COALESCE(CAST(brightness_nits_scraped AS DOUBLE), 0.0) / 300.0 +
    COALESCE(CAST(reviews_count AS DOUBLE), 0.0) / 1000.0
  ) AS performance_score

FROM v_kabum_features_scored
WHERE price IS NOT NULL
  AND (rating IS NOT NULL OR refresh_rate_hz_scraped IS NOT NULL OR brightness_nits_scraped IS NOT NULL)
ORDER BY performance_score DESC
LIMIT 50;


product_name,brand,price,rating,reviews_count,refresh_rate_hz_scraped,brightness_nits_scraped,performance_score
Notebook Acer Predator Phn16-72-99my Intel Ci9 14900hx 14ª Gen 32gb 1TB SSD RTX 4070 W11 Home - NH.QTVAL.002,Acer,13499.1,5.0,16,240.0,500.0,10.682666666666666
Notebook Acer Predator Phn16-72-99my Intel Ci9 14900hx 14ª Gen 32gb 1TB SSD RTX 4070 W11 Home - NH.QTVAL.002,Acer,13499.1,5.0,16,240.0,500.0,10.682666666666666
"Notebook Gamer Acer Nitro V15, AMD Ryzen 7-7735HS, 16GB RAM, RTX 4050, SSD 512 GB, Tela 15.6"" Full HD, Linux 64 Bits - Anv15-41-R2gt",Acer,5760.0,5.0,21,165.0,300.0,8.771
"Notebook Gamer Acer Nitro V15, AMD Ryzen 7-7735HS, 16GB RAM, RTX 4050, SSD 512 GB, Tela 15.6"" Full HD, Linux 64 Bits - Anv15-41-R2gt",Acer,5760.0,5.0,21,165.0,300.0,8.771
"Notebook Gamer Acer Nitro V15, AMD Ryzen 7-7735HS, 16GB RAM, RTX 4050, SSD 512 GB, Tela 15.6"" Full HD, Linux 64 Bits - Anv15-41-R2gt",Acer,5760.0,5.0,21,165.0,300.0,8.771
"Notebook Gamer Acer Nitro V15, AMD Ryzen 7-7735HS, 16GB RAM, RTX 4050, SSD 512 GB, Tela 15.6"" Full HD, Linux 64 Bits - Anv15-41-R2gt",Acer,5760.0,5.0,21,165.0,300.0,8.771
"Notebook Gamer Acer Nitro V15, AMD Ryzen 7-7735HS, 16GB RAM, RTX 4050, SSD 512 GB, Tela 15.6"" Full HD, Linux 64 Bits - Anv15-41-R2gt",Acer,5760.0,5.0,21,165.0,300.0,8.771
"Notebook Gamer Acer Nitro V15, AMD Ryzen 7-7735HS, 16GB RAM, RTX 4050, SSD 512 GB, Tela 15.6"" Full HD, Linux 64 Bits - Anv15-41-R2gt",Acer,5760.0,5.0,21,165.0,300.0,8.771
"Notebook Gamer Acer Nitro V15, Intel Core i7-13620h 13ªG, 16GB RAM, RTX 4050, SSD 512GB, Tela 15.6"" Full HD, Linux - Anv15-52-737p",Acer,6908.01,5.0,15,165.0,300.0,8.765
"Notebook Gamer Acer Nitro V15, Intel Core i7-13620h 13ªG, 16GB RAM, RTX 4050, SSD 512GB, Tela 15.6"" Full HD, Linux - Anv15-52-737p",Acer,6908.01,5.0,15,165.0,300.0,8.765


Price segmentation (Budget / Mid / Premium)

Segmentação geral

In [0]:
%sql
WITH base AS (
  SELECT
    *,
    CASE
      WHEN price < 3500 THEN 'BUDGET (<3.5k)'
      WHEN price < 7000 THEN 'MID (3.5k–7k)'
      WHEN price < 12000 THEN 'PREMIUM (7k–12k)'
      ELSE 'ULTRA (12k+)'
    END AS price_segment
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL
)
SELECT
  price_segment,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_products,
  ROUND(AVG(price), 2) AS avg_price,
  ROUND(percentile_approx(price, 0.5), 2) AS median_price
FROM base
GROUP BY price_segment
ORDER BY
  CASE price_segment
    WHEN 'BUDGET (<3.5k)' THEN 1
    WHEN 'MID (3.5k–7k)' THEN 2
    WHEN 'PREMIUM (7k–12k)' THEN 3
    ELSE 4
  END;

price_segment,products,pct_products,avg_price,median_price
BUDGET (<3.5k),40,20.00,2970.78,2999.0
MID (3.5k–7k),92,46.00,5016.77,5199.0
PREMIUM (7k–12k),22,11.00,8836.03,8099.1
ULTRA (12k+),46,23.00,25984.54,24400.0


Segmentação por marca (market positioning)

In [0]:
%sql
WITH base AS (
  SELECT
    brand,
    price,
    CASE
      WHEN price < 3500 THEN 'BUDGET'
      WHEN price < 7000 THEN 'MID'
      WHEN price < 12000 THEN 'PREMIUM'
      ELSE 'ULTRA'
    END AS price_segment
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL
)
SELECT
  brand,
  price_segment,
  COUNT(*) AS products,
  ROUND(AVG(price),2) AS avg_price
FROM base
GROUP BY brand, price_segment
ORDER BY brand, price_segment;

brand,price_segment,products,avg_price
ASUS,BUDGET,16,2973.71
ASUS,MID,18,5266.82
ASUS,PREMIUM,8,9096.24
ASUS,ULTRA,20,29314.7
Acer,BUDGET,4,3199.05
Acer,MID,38,5205.06
Acer,PREMIUM,8,7902.7
Acer,ULTRA,6,14999.1
Apple,PREMIUM,2,11999.0
Dell,MID,2,3689.1


Brand market share (share do catálogo)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS market_share_pct,
  ROUND(AVG(price),2) AS avg_price,
  ROUND(percentile_approx(price,0.5),2) AS median_price
FROM v_kabum_features_scored
WHERE brand IS NOT NULL
GROUP BY brand
ORDER BY products DESC;

brand,products,market_share_pct,avg_price,median_price
ASUS,62,31.00,12926.55,6447.0
Acer,56,28.00,6496.51,5760.0
Lenovo,44,22.00,4280.74,3999.0
Samsung,8,4.00,20523.51,17000.0
MSI,8,4.00,30950.0,30900.0
VAIO,6,3.00,3877.4,3284.1
Microsoft,4,2.00,21550.0,18700.0
Hp,4,2.00,2806.25,2199.0
Gigabyte,4,2.00,13149.5,9999.0
Dell,2,1.00,3689.1,3689.1


Share por segmento (quem domina Budget/Mid/Premium)

In [0]:
%sql
WITH base AS (
  SELECT
    brand,
    price,
    CASE
      WHEN price < 3500 THEN 'BUDGET'
      WHEN price < 7000 THEN 'MID'
      WHEN price < 12000 THEN 'PREMIUM'
      ELSE 'ULTRA'
    END AS price_segment
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL AND brand IS NOT NULL
),
seg_tot AS (
  SELECT price_segment, COUNT(*) AS seg_total
  FROM base
  GROUP BY price_segment
)
SELECT
  b.price_segment,
  b.brand,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / t.seg_total, 2) AS share_within_segment_pct
FROM base b
JOIN seg_tot t USING (price_segment)
GROUP BY b.price_segment, b.brand, t.seg_total
ORDER BY b.price_segment, products DESC;

price_segment,brand,products,share_within_segment_pct
BUDGET,ASUS,16,40.00
BUDGET,Lenovo,12,30.00
BUDGET,VAIO,4,10.00
BUDGET,Hp,4,10.00
BUDGET,Acer,4,10.00
MID,Acer,38,41.30
MID,Lenovo,30,32.61
MID,ASUS,18,19.57
MID,Dell,2,2.17
MID,Samsung,2,2.17


In [0]:
%sql
DESCRIBE v_kabum_features_scored;

col_name,data_type,comment
product_key,string,null
ingestion_date,date,null
marketplace,string,null
search_term,string,null
kabum_code,bigint,null
product_name,string,null
brand_id,bigint,null
brand,string,null
brand_img,string,null
category,string,null


In [0]:
%sql
SELECT * FROM v_kabum_features_scored LIMIT 5;

product_key,ingestion_date,marketplace,search_term,kabum_code,product_name,brand_id,brand,brand_img,category,available,currency,price,old_price,discount_pct,rating,reviews_count,max_installment,product_url,image_url,scraped_at,price_raw,old_price_raw,discount_pct_raw,discount_pct_site,discount_pct_gap,product_url_scraped,refresh_rate_hz_scraped,screen_resolution_scraped,panel_type_scraped,panel_finish_scraped,brightness_nits_scraped,scrape_ok_scraped,scrape_error_scraped,enriched_from_scrape,catalog_fields_total,catalog_fields_filled,catalog_completeness_pct,catalog_completeness_score,catalog_completeness_tier
0005fe5fae3c3d002f547418ceedfe52a766c420c9ece022d716a04f43e0798d,2026-03-05,kabum,notebook,883577,"Notebook Lenovo Ideapad Slim 3 15irh10 Intel Core i5-13420h 16gb 512gb SSD WINDOWS 11 15.3"" - 83ns0001br Luna Grey",205,Lenovo,https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg,Computadores/Notebooks/Notebook Lenovo,true,BRL,3695.12,null,0,5.0,3,"10x de R$ 419,90",https://www.kabum.com.br/produto/883577/notebook-lenovo-ideapad-slim-3-15irh10-intel-core-i5-13420h-16gb-512gb-ssd-windows-11-15-3-83ns0001br-luna-grey,https://images.kabum.com.br/produtos/fotos/sync_mirakl/883577/xlarge/Notebook-Lenovo-Ideapad-Slim-3-15irh10-Intel-Core-i5-13420h-16gb-512gb-SSD-WINDOWS-11-15-3-83ns0001br-Luna-Grey_1770929879.jpg,2026-03-04T21:24:48.619Z,null,null,12.0,12.0,12.0,https://www.kabum.com.br/produto/883577/notebook-lenovo-ideapad-slim-3-15irh10-intel-core-i5-13420h-16gb-512gb-ssd-windows-11-15-3-83ns0001br-luna-grey,60.0,1920x1200,IPS,MATTE,300.0,true,null,true,3,3,100.0,3,HIGH
0461b3e06c104ca6956d220d00e9ae48fb5a8d6502573d38e450799e45eb03b7,2026-03-05,kabum,notebook,627471,Notebook Asus Vivobook 16 X1605va Intel Core I7 1355u 16gb Ram 512gb SSD WINDOWS 11 Home Tela 16” Ips Fhd Silver - Mb763w,1,ASUS,https://images4.kabum.com.br/produtos/fabricantes/logo-asus.jpg,Computadores/Notebooks/Notebook Asus,true,BRL,4304.81,4836.87,11,5.0,1,"10x de R$ 435,31",https://www.kabum.com.br/produto/627471/notebook-asus-vivobook-16-x1605va-intel-core-i7-1355u-16gb-ram-512gb-ssd-windows-11-home-tela-16-ips-fhd-silver-mb763w,https://images.kabum.com.br/produtos/fotos/sync_mirakl/627471/xlarge/Notebook-Asus-Vivobook-16-X1605va-Intel-Core-I7-1355u-16gb-Ram-512gb-SSD-WINDOWS-11-Home-Tela-16-Ips-Fhd-Silver-Mb763w_1765585717.jpg,2026-03-04T21:24:48.619Z,null,null,11.0,11.0,0.0,https://www.kabum.com.br/produto/627471/notebook-asus-vivobook-16-x1605va-intel-core-i7-1355u-16gb-ram-512gb-ssd-windows-11-home-tela-16-ips-fhd-silver-mb763w,null,1920x1200,IPS,null,null,true,null,true,3,3,100.0,3,HIGH
18c57d400cb2cf0d50d171179055b0cbccbda123bbd16e26be37018064510784,2026-03-05,kabum,notebook,879301,"Notebook Lenovo Loq-e 15iax9e Intel Core i7-12650hx 16gb 512gb SSD RTX 4050 Linux 15.6"" - 83mes00200 Luna Grey",205,Lenovo,https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg,Computadores/Notebooks/Notebook Lenovo,true,BRL,7202.6,null,0,5.0,2,"10x de R$ 827,88",https://www.kabum.com.br/produto/879301/notebook-lenovo-loq-e-15iax9e-intel-core-i7-12650hx-16gb-512gb-ssd-rtx-4050-linux-15-6-83mes00200-luna-grey,https://images.kabum.com.br/produtos/fotos/sync_mirakl/879301/xlarge/Notebook-Lenovo-Loq-e-15iax9e-Intel-Core-i7-12650hx-16gb-512gb-SSD-RTX-4050-Linux-15-6-83mes00200-Luna-Grey_1765465529.jpg,2026-03-04T21:24:48.619Z,null,null,13.0,13.0,13.0,https://www.kabum.com.br/produto/879301/notebook-lenovo-loq-e-15iax9e-intel-core-i7-12650hx-16gb-512gb-ssd-rtx-4050-linux-15-6-83mes00200-luna-grey,144.0,1920x1080,IPS,MATTE,300.0,true,null,true,3,3,100.0,3,HIGH
2c67f0724aea643dd2eb457186d9d7c64d7b513210f393ddf877ad5ed3621866,2026-03-05,kabum,notebook,728904,"Notebook Lenovo LOQ-E Core i5-12450HX, RTX 3050, 16GB, 512GB SSD, 15.6"" FHD, IPS, 144Hz, Win 11, Luna Cinza- 83ME0007BR",205,Lenovo,https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg,Computadores/Notebooks/Notebook Lenovo,true,BRL,5947.53,null,0,5.0,73,"10x de R$ 660,83",https://www